# Portfolio Analysis

## What you'll learn
- How assets move together (correlation)
- Combining assets into a weighted portfolio
- Comparing the portfolio to its parts

In [ ]:
import pandas as pd

df = pd.read_csv("data/stock_history.csv", parse_dates=["Date"])
df = df.drop_duplicates().sort_values("Date")
# One price column per ticker, indexed by date:
closes = df.pivot_table(index="Date", columns="Ticker", values="Close")
closes = closes.ffill()   # carry the last known price over any gaps
print(closes.head())

In [ ]:
returns = closes.pct_change().dropna()
print(returns.head())

## Correlation

Diversification works when assets do **not** all move together. `corr()` measures that:
1.0 means they move in lockstep, 0 means unrelated, negative means they tend to move in
opposite directions.

In [ ]:
print(returns.corr().round(2))

## Building a portfolio

A portfolio assigns a **weight** (share of money) to each asset; the weights add to 1. The
portfolio's daily return is the weighted average of the individual returns. `mul(..., axis=1)`
multiplies each column by its weight, and `sum(axis=1)` adds across the row.

In [ ]:
import pandas as pd

weights = pd.Series({"ATLAS": 0.4, "HELIOS": 0.2, "NORTH": 0.3, "VERTEX": 0.1})
print("weights sum to", weights.sum())

portfolio_returns = returns.mul(weights, axis=1).sum(axis=1)
print(portfolio_returns.head())

## Portfolio growth and risk

Treat the portfolio's daily returns exactly like a single asset: compound them for growth,
take their standard deviation for risk.

In [ ]:
import numpy as np

growth = (1 + portfolio_returns).cumprod()
print("Final value of 100 invested:", round(growth.iloc[-1] * 100, 1))
print("Total return:", round((growth.iloc[-1] - 1) * 100, 1), "%")
print("Annualised volatility:", round(portfolio_returns.std() * np.sqrt(252) * 100, 1), "%")

## Portfolio vs its parts

Plot the portfolio's growth alongside each asset. The point of mixing is usually a smoother
ride than the riskiest single holding.

In [ ]:
import matplotlib.pyplot as plt

asset_growth = (1 + returns).cumprod()
ax = asset_growth.plot(figsize=(9, 5), alpha=0.5, title="Growth of 1 unit: assets vs portfolio")
growth.plot(ax=ax, label="PORTFOLIO", linewidth=2.5, color="black")
ax.set_ylabel("Growth factor")
ax.legend()
plt.tight_layout()
plt.show()

## Practice

1. Build clean wide `closes` and daily `returns`.
2. Print the correlation matrix and name the pair that moves together most.
3. Build an equally weighted portfolio (0.25 each) and compute its daily returns.
4. Compute the portfolio's total return and annualised volatility.
5. Plot the equally weighted portfolio's growth against the individual assets.

In [ ]:
# Your practice code here

## Summary

- `returns.corr()` shows how assets move together; low correlation aids diversification.
- A portfolio weights each asset (weights sum to 1); its return is the weighted average.
- `returns.mul(weights, axis=1).sum(axis=1)` computes daily portfolio returns.
- Analyse the portfolio like any single series: compound for growth, std for risk.

## Practice Solutions

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("data/stock_history.csv", parse_dates=["Date"]).drop_duplicates().sort_values("Date")
closes = df.pivot_table(index="Date", columns="Ticker", values="Close").ffill()
returns = closes.pct_change().dropna()

# 2.
print(returns.corr().round(2))

# 3.
weights = pd.Series({"ATLAS": 0.25, "HELIOS": 0.25, "NORTH": 0.25, "VERTEX": 0.25})
port = returns.mul(weights, axis=1).sum(axis=1)

# 4.
growth = (1 + port).cumprod()
print("Total return:", round((growth.iloc[-1] - 1) * 100, 1), "%")
print("Annual volatility:", round(port.std() * np.sqrt(252) * 100, 1), "%")

# 5.
ax = (1 + returns).cumprod().plot(figsize=(9, 5), alpha=0.5)
growth.plot(ax=ax, label="PORTFOLIO", linewidth=2.5, color="black")
ax.legend()
plt.tight_layout()
plt.show()